In [11]:
import pandas as pd

In [12]:
training_master_raw = pd.read_csv(
    "../Data/processed/training_load/training_load_master.csv"
)

training_master_raw.head()

,Date,player_id,acwr,atl,ctl28,ctl42,daily_load,monotony,strain
0,01.01.2020,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,01.01.2020,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,01.01.2020,TeamA-32fed4b3-d7fc-482d-ba21-c46c58f015b5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,01.01.2020,TeamA-358603ef-b3a3-46b5-b80a-ad64e06b6592,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,01.01.2020,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
game_performance = pd.read_csv(
    "../Data/subjective/game-performance/game-performance.csv"
)

game_performance.head()

,player_name,team_performance,offensive_performance,defensive_performance,timestamp
0,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,7,5,6,11.07.2020
1,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,7,7,7,07.10.2020
2,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,5,7,5,18.10.2020
3,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,6,6,7,31.10.2020
4,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,4,4,4,07.11.2020


Check weather Game Performance and Training Data Matches or not

In [14]:
try:
    training_players = set(training_master["player_id"].unique())
except NameError:
    try:
        training_master = pd.read_csv("../Data/subjective/training-master/training-master.csv")
    except FileNotFoundError:
        # fallback: build a minimal training_master from game_performance.player_name
        training_master = pd.DataFrame({"player_id": game_performance["player_name"].unique()})
        print("Warning: training-master.csv not found. Created training_master from game_performance.player_name")
    training_players = set(training_master["player_id"].unique())

game_players = set(game_performance["player_name"].unique())

print("Training Players:", len(training_players))
print("Game Players:", len(game_players))

print("Common Players:", len(training_players.intersection(game_players)))
game_players = set(game_performance["player_name"].unique())

print("Training Players:", len(training_players))
print("Game Players:", len(game_players))

print("Common Players:", len(training_players.intersection(game_players)))

Training Players: 36
Game Players: 36
Common Players: 36
Training Players: 36
Game Players: 36
Common Players: 36


Conclusion

✅ All 36 players exist in both datasets

✅ Player IDs match perfectly

✅ No player mapping table needed

Next Check: Dates

Players match.

Now we need to check whether the dates overlap.

In [18]:
training_master = training_master_raw.copy()

training_master["Date"] = pd.to_datetime(
    training_master["Date"],
    format="%d.%m.%Y",
    errors="coerce"
)

if training_master["Date"].notna().any():
    print(
        "Training Date Range:",
        training_master["Date"].min(),
        "to",
        training_master["Date"].max()
    )
else:
    print("Training Date Range: unavailable (no valid Date column)")

print(
    "Game Date Range:",
    game_performance["timestamp"].min(),
    "to",
    game_performance["timestamp"].max()
)

game_performance["timestamp"] = pd.to_datetime(
    game_performance["timestamp"],
    format="%d.%m.%Y"
)

print(
    "Training Date Range:",
    training_master["Date"].min(),
    "to",
    training_master["Date"].max()
)

print(
    "Game Date Range:",
    game_performance["timestamp"].min(),
    "to",
    game_performance["timestamp"].max()
)

Training Date Range: 2020-01-01 00:00:00 to 2021-12-31 00:00:00
Game Date Range: 2020-06-20 00:00:00 to 2021-11-17 00:00:00
Training Date Range: 2020-01-01 00:00:00 to 2021-12-31 00:00:00
Game Date Range: 2020-06-20 00:00:00 to 2021-11-17 00:00:00


In [17]:
training_master_raw = pd.read_csv(
    "../Data/processed/training_load/training_load_master.csv"
)

training_master_raw.head()

,Date,player_id,acwr,atl,ctl28,ctl42,daily_load,monotony,strain
0,01.01.2020,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,01.01.2020,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,01.01.2020,TeamA-32fed4b3-d7fc-482d-ba21-c46c58f015b5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,01.01.2020,TeamA-358603ef-b3a3-46b5-b80a-ad64e06b6592,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,01.01.2020,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Merge Performance Data

In [19]:
training_master = training_master.rename(
    columns={
        "Date": "date",
        "player_id": "player_name"
    }
)

game_performance = game_performance.rename(
    columns={
        "timestamp": "date"
    }
)

In [20]:
print(training_master["date"].dtype)
print(game_performance["date"].dtype)

datetime64[us]
datetime64[us]


In [21]:
performance_master = training_master.merge(
    game_performance,
    on=["player_name", "date"],
    how="left"
)

In [22]:
performance_master[
    [
        "team_performance",
        "offensive_performance",
        "defensive_performance"
    ]
].notna().sum()

team_performance         248
offensive_performance    248
defensive_performance    248
dtype: int64